## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-19-09-49-23 +0000,nyt,Live Updates: Trump Holds First Summit With Xi...,https://www.nytimes.com/live/2026/11/18/world/...
1,2026-11-19-09-47-00 +0000,wsj,Stock Futures Rise as Optimism Improves Ahead ...,https://www.wsj.com/finance/stocks/stock-futur...
2,2026-11-19-09-44-35 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/19/world/...
3,2026-11-19-09-40-00 +0000,wsj,"Treasury, Eurozone Bond Yields Edge Higher",https://www.wsj.com/economy/central-banking/tr...
4,2026-11-19-09-37-00 +0000,wsj,Houthi Attack Damages Saudi Capital’s Tallest ...,https://www.wsj.com/world/middle-east/houthi-a...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2756/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
2,trump,63
6,xi,21
7,putin,21
355,new,20
154,china,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
18,2026-11-19-09-10-11 +0000,startribune,Nuclear arms and ending wars in Iran and Ukrai...,https://www.startribune.com/trump-meets-xi-and...,168
298,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...,166
71,2026-11-19-03-53-50 +0000,chitri,President Trump meets Xi and Putin in China wi...,https://www.chicagotribune.com/2026/11/18/trum...,163
398,2026-11-18-12-07-00 +0000,wsj,"In Choosing Shenzhen for Trump-Putin Summit, X...",https://www.wsj.com/world/china/in-choosing-sh...,152
221,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...,152


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
18,168,2026-11-19-09-10-11 +0000,startribune,Nuclear arms and ending wars in Iran and Ukrai...,https://www.startribune.com/trump-meets-xi-and...
101,109,2026-11-19-02-00-00 +0000,wsj,Jensen Huang’s ouster from Nvidia is giving ne...,https://www.wsj.com/politics/national-security...
116,83,2026-11-19-01-11-00 +0000,wsj,Elon Musk said he felt “pretty foolish” about ...,https://www.wsj.com/politics/policy/elon-musk-...
221,83,2026-11-18-21-39-00 +0000,wsj,A top House Democrat issued document-preservat...,https://www.wsj.com/politics/policy/democrats-...
195,68,2026-11-18-22-13-12 +0000,cbc,She says Russell Brand sexually assaulted her ...,https://www.cbc.ca/radio/asithappens/mia-latte...
437,53,2026-11-18-10-23-57 +0000,bbc,Democrats win Alaska Senate race two weeks aft...,https://www.bbc.co.uk/news/articles/cqev3v2ygz...
92,52,2026-11-19-02-20-15 +0000,bbc,Anthropic valued at $1.5 trillion ahead of AI ...,https://www.bbc.co.uk/news/articles/c6742m39xg...
139,50,2026-11-19-00-09-28 +0000,chitri,"Illinois Supreme Court picks Jennifer Soble, I...",https://www.chicagotribune.com/2026/11/18/jenn...
398,47,2026-11-18-12-07-00 +0000,wsj,"In Choosing Shenzhen for Trump-Putin Summit, X...",https://www.wsj.com/world/china/in-choosing-sh...
155,45,2026-11-18-23-23-06 +0000,nyt,"NYC, LA and Chicago School Districts Weigh Bre...",https://www.nytimes.com/2026/11/18/technology/...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
